# Natural Language Processing with Python

In this workshop we will use the [Sentiment Labelled Sentences Data Set](https://archive.ics.uci.edu/ml/datasets/Sentiment+Labelled+Sentences)  that contains 500 positive and 500 negative sentences from imdb.com, amazon.com and yelp.com.  

* First, we will explore preliminary text analytics and text pre-processing .  
* Second, we will evaluate different feature extraction mechanisms for text.  
* Third, we will evaluate a simple text classification for Amazon review dataset, and an advanced deep neural network for classification accuracy improvement.


## Load the data

Load the reviews.csv file downloaded from LMS to the Google Colab file repository.

In [1]:
import pandas as pd

In [2]:
# Dataset reveiws.csv MUST be uploaded to Google Colab before executing this line
df = pd.read_csv('tp6_reviews.csv')

View a summary of the dataset.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2748 entries, 0 to 2747
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sentence  2748 non-null   str  
 1   label     2748 non-null   int64
 2   source    2748 non-null   str  
dtypes: int64(1), str(2)
memory usage: 64.5 KB


In [4]:
df.head()

,sentence,label,source
0,Wow... Loved this place.,1,yelp
1,Crust is not good.,0,yelp
2,Not tasty and the texture was just nasty.,0,yelp
3,Stopped by during the late May bank holiday of...,1,yelp
4,The selection on the menu was great and so wer...,1,yelp


Check what are the sources of data available in the dataset.

In [5]:
df['source'].unique()

<StringArray>
['yelp', 'amazon', 'imdb']
Length: 3, dtype: str

## Preliminary analysis

***Pandas dataframe's df.apply() function***  
* This allow the users to pass a function and apply it on every single value of the Pandas series, i.e., column. [API documentation.](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.apply.html)
*  Efficient way to update values in a dataframe column

In the following example,


*   Count the number of words in each sentence
*   Assign the word count to a new attribute  named 'word_count'



In [6]:
# Count the number of words in each sentence
# Assign the word count to a new attribute  named 'word_count'

def word_counter(sentence):
  split_word = str(sentence).split(" ") # split by white space to get words list
  word_count = len(split_word) # count the words in the list
  return word_count

df['word_count'] = df['sentence'].apply(word_counter)
df.head(5)

,sentence,label,source,word_count
0,Wow... Loved this place.,1,yelp,4
1,Crust is not good.,0,yelp,4
2,Not tasty and the texture was just nasty.,0,yelp,8
3,Stopped by during the late May bank holiday of...,1,yelp,15
4,The selection on the menu was great and so wer...,1,yelp,12


Similarly, count the number of characters of each sentence

In [7]:
df['char_count'] = df['sentence'].str.len()  # Includes the spaces
df.head(5)

,sentence,label,source,word_count,char_count
0,Wow... Loved this place.,1,yelp,4,24
1,Crust is not good.,0,yelp,4,18
2,Not tasty and the texture was just nasty.,0,yelp,8,41
3,Stopped by during the late May bank holiday of...,1,yelp,15,87
4,The selection on the menu was great and so wer...,1,yelp,12,59


Calculate the average word length for each sentence.

*   First, construct a method (avg_word()) which takes a sentence, split the sentence to words, then calculate the average word length.
*   Using pandas dataframe apply() function and avg_word() method, calculate the average word length
*  Assign the value to new column names 'avg_word'



In [8]:
# Average word length
def avg_word(sentence):
  words = sentence.split(" ") # split by white space to get words list
  all_char_count = sum(len(word) for word in words) # count all characters
  avg_of_words = all_char_count/len(words) # Summation of characters in all words / number of words
  return avg_of_words

df['avg_word'] = df['sentence'].apply(avg_word)
df.head(5)

,sentence,label,source,word_count,char_count,avg_word
0,Wow... Loved this place.,1,yelp,4,24,5.250000
1,Crust is not good.,0,yelp,4,18,3.750000
2,Not tasty and the texture was just nasty.,0,yelp,8,41,4.250000
3,Stopped by during the late May bank holiday of...,1,yelp,15,87,4.866667
4,The selection on the menu was great and so wer...,1,yelp,12,59,4.000000


## Text pre-processing

Pre-processing is mandatory for most text analytics tasks, as text in its raw format is unstructured and noisy.

In the following snippets you will run several pre-processing steps.  

**Please note that pre-processing is to be used with clear understanding of the expected outcome of text analytics, as each pre-processing step is not relevant or applicable to every NLP task.**

Uppercase and lowercase characters are used for clarity in human communication. However, for a machine such distinction would create unnecessary complexities. Therefore, we transform all characters to lowercase.

In [9]:
df['sentence'] = df['sentence'].str.lower()
df.head()

,sentence,label,source,word_count,char_count,avg_word
0,wow... loved this place.,1,yelp,4,24,5.250000
1,crust is not good.,0,yelp,4,18,3.750000
2,not tasty and the texture was just nasty.,0,yelp,8,41,4.250000
3,stopped by during the late may bank holiday of...,1,yelp,15,87,4.866667
4,the selection on the menu was great and so wer...,1,yelp,12,59,4.000000


Same with punctuation marks, we remove all using [regular expressions](https://www.w3schools.com/python/python_regex.asp) .  
**Regular Expressions** - A regular expression is a special sequence of characters that helps you match or find other strings or sets of strings.

In [10]:
# This regular expression only keeps words and characters
df['sentence'] = df['sentence'].str.replace(r'[^\w\s]','', regex=True)
df.head()

,sentence,label,source,word_count,char_count,avg_word
0,wow loved this place,1,yelp,4,24,5.250000
1,crust is not good,0,yelp,4,18,3.750000
2,not tasty and the texture was just nasty,0,yelp,8,41,4.250000
3,stopped by during the late may bank holiday of...,1,yelp,15,87,4.866667
4,the selection on the menu was great and so wer...,1,yelp,12,59,4.000000


### Remove digits

For a sentiment analytics task, numbers or digits are not needed. Thus, we remove digits from the text dataset.

However, for other tasks, numbers may be needed.

In [11]:
def remove_digits(sentence):
  word_list = sentence.split(" ")
  words_without_numbers = [word for word in word_list if not word.isdigit()]
  return " ".join(words_without_numbers)

df['sentence'] = df['sentence'].apply(remove_digits)
df.head()

,sentence,label,source,word_count,char_count,avg_word
0,wow loved this place,1,yelp,4,24,5.250000
1,crust is not good,0,yelp,4,18,3.750000
2,not tasty and the texture was just nasty,0,yelp,8,41,4.250000
3,stopped by during the late may bank holiday of...,1,yelp,15,87,4.866667
4,the selection on the menu was great and so wer...,1,yelp,12,59,4.000000


Demonstrate of the remove digit function

In [12]:
sample_text = 'Covid 19 is spreading fast'
print(remove_digits(sample_text))

Covid is spreading fast


What does "".join() means?

In [13]:
word_list = ["Covid", "is", "spreading", "fast"]
sentence = "     ".join(word_list)
print(sentence)

Covid     is     spreading     fast


### Remove Stopwords

[Stopwords](https://en.wikipedia.org/wiki/Stop_words) are deemed irrelevant for NLP purposes because they occur frequently in the language. Therefore, we will omit the stopwords as a pre-processing step. For this, we will use [NLTK](https://www.nltk.org/) library here.

**NLTK, is a suite of libraries and programs for symbolic and statistical natural language processing (NLP) specifically for the English language written in  Python.**

In [14]:
# Load NLTK library
import nltk

# Download the stopwords to the nltk library
nltk.download('stopwords')

# Load the stopwords
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\AMatharaArac/nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Have a look at the stopwords indexed in the NLTK library.

In [15]:
stop = stopwords.words('english')
print(stop)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Remove unwanted stop words from the NLTK stop word list.

In [16]:
all_words_i_want = ['not', 'had', "has"]
for w in all_words_i_want:
  stop.remove(w)

Remove stopwords from the sentences.

In [17]:
def remove_stopwords(sentence):
  words_without_stopwords = [word for word in sentence.split(" ") if word not in stop]
  return " ".join(words_without_stopwords)

df['sentence'] = df['sentence'].apply(remove_stopwords)
df.head(5)

,sentence,label,source,word_count,char_count,avg_word
0,wow loved place,1,yelp,4,24,5.250000
1,crust not good,0,yelp,4,18,3.750000
2,not tasty texture nasty,0,yelp,8,41,4.250000
3,stopped late may bank holiday rick steve recom...,1,yelp,15,87,4.866667
4,selection menu great prices,1,yelp,12,59,4.000000


### Common and rare word analysis

Aside from stopwords, some words appear rarely (only once or twice) in an entire body of text.
Based on the analytics requirement, you can decide whether to keep or remove, and at what intensity/scale to remove.

In order to do this, first we have to construct a word frequency dictionary.

In [18]:
# First we have to construct a word frequency dictionary.

full_document = ' '.join(df['sentence']) # combine all sentences together
all_words_list = full_document.split(" ") # extract all words to a list by splitting by spaces
word_frequency = pd.Series(all_words_list).value_counts() # count words

print(word_frequency)

                 1902
not               304
good              225
great             205
movie             177
                 ... 
jessice             1
clothes             1
virtue              1
regrettable         1
exceptionally       1
Name: count, Length: 5318, dtype: int64


List the top 10 common words.

In [19]:
# Top common words
word_frequency[:10]  # get top 10

         1902
not       304
good      225
great     205
movie     177
phone     162
film      155
one       141
had       138
like      123
Name: count, dtype: int64

List the top 10 rare words.

In [20]:
# least common words
word_frequency[-10:]  # get top 10

flowed           1
malebonding      1
hoot             1
n                1
genre            1
jessice          1
clothes          1
virtue           1
regrettable      1
exceptionally    1
Name: count, dtype: int64

### Spelling correction

To correct misspelt words, we will use [textblob library](https://textblob.readthedocs.io/en/dev/) library. Keep in mind that corrections are always bound by the dictionary that you would use, and it may not account for context (their vs there).

Due to the time complexity of spell-checking an entire corpus, in this exercise, we will use spell-check for just one example.

In [21]:
from textblob import TextBlob

In [22]:
# Do not run this line of code.
# Following line of code will correct spellings of all the sentences in the dataset.
# df['sentence'] = df['sentence'].apply(lambda x: str(TextBlob(x).correct()))   # This will take a long time. Thus, we will show an seperate example

Spelling correction example

In [23]:
corrected = str(TextBlob('bisness').correct())

print(corrected)

business


In [24]:
# Keep in mind that corrections are always bound by the dictionary that you would use, and it may not account for context (their vs there).

incorrect_text = 'bisness anlytis is an itant skil seit for any organizaton'

corrected_text = str(TextBlob(incorrect_text).correct())
print(corrected_text)

business analysis is an want skin set for any organization


### Stemming

[Stemming](https://en.wikipedia.org/wiki/Stemming) is the removal of prefix, suffix etc, to derive the base form of a word. We will use the NLTK library.

In [25]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stemming_function(sentence):
  word_list = sentence.split(" ")
  stemmed_word_list = [stemmer.stem(word) for word in word_list]
  stemmed_sentence = " ".join(stemmed_word_list)
  return stemmed_sentence

df['sentence_stemmed'] = df['sentence'].apply(stemming_function)

df.head()

,sentence,label,source,word_count,char_count,avg_word,sentence_stemmed
0,wow loved place,1,yelp,4,24,5.250000,wow love place
1,crust not good,0,yelp,4,18,3.750000,crust not good
2,not tasty texture nasty,0,yelp,8,41,4.250000,not tasti textur nasti
3,stopped late may bank holiday rick steve recom...,1,yelp,15,87,4.866667,stop late may bank holiday rick steve recommen...
4,selection menu great prices,1,yelp,12,59,4.000000,select menu great price


### Lemmatization

[Lemmatization](https://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html), unlike Stemming, reduces the inflected words properly ensuring that the root word belongs to the language. In Lemmatization root word is called lemma. A lemma (plural lemmas or lemmata) is the canonical form, dictionary form, or citation form of a set of words.  
  
We will use  Wordnet for the lemmatization. Thus, we need to download Wordnet to the nltk library.

WordNet is a lexical database for the English language. It groups English words into sets of synonyms called synsets, provides short definitions and usage


In [26]:
# Download wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\AMatharaArac/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\AMatharaArac/nltk_data...


True

In [27]:
from nltk.stem import WordNetLemmatizer

lemmtizer = WordNetLemmatizer()

In [28]:
def lemmatize_function(sentence, pos='n'):
  word_list = sentence.split()
  # pos = 'n' for nouns, 'v' for verbs ...
  lemma_word_list = [lemmtizer.lemmatize(word, pos=pos) for word in word_list]
  lemma_sentence = " ".join(lemma_word_list)
  return lemma_sentence

df['sentence_lemmatized'] = df['sentence'].apply(lemmatize_function)

In [29]:
lemmatize_function("I cannot believe dogs are walking", pos='v')

'I cannot believe dog be walk'

Display original pre-processed sentence, stemmed sentence and lemmatized sentence.

In [30]:
df[['sentence', 'sentence_stemmed', 'sentence_lemmatized']].head(10)

,sentence,sentence_stemmed,sentence_lemmatized
0,wow loved place,wow love place,wow loved place
1,crust not good,crust not good,crust not good
2,not tasty texture nasty,not tasti textur nasti,not tasty texture nasty
3,stopped late may bank holiday rick steve recom...,stop late may bank holiday rick steve recommen...,stopped late may bank holiday rick steve recom...
4,selection menu great prices,select menu great price,selection menu great price
5,getting angry want damn pho,get angri want damn pho,getting angry want damn pho
6,honeslty didnt taste fresh,honeslti didnt tast fresh,honeslty didnt taste fresh
7,potatoes like rubber could tell had made ahead...,potato like rubber could tell had made ahead t...,potato like rubber could tell had made ahead t...
8,fries great,fri great,fry great
9,great touch,great touch,great touch


Stemmed algorithm seems to be working better in this case when compared to the lemmatization. It is recommended to observe the results after pre-processing tasks to understand the performance of the third party libraries we are using in pre-processing.

## Text Feature Extraction

In a numeric dataset (e.g., house price dataset, titanic survival dataset, dungaree dataset), we had numeric and categorical variables, which we transformed to numeric values for predictive analytics. Those numeric variables are called numeric features in the datasets. Similarly, for NLP, we need to derive features from text data in numerical format because machines can only understand numeric representations.

### N-Grams

An [n-gram](https://en.wikipedia.org/wiki/N-gram) is a contiguous sequence of n items from a given sample of text or speech. They are basically a set of co-occuring words within a given window. When computing the n-grams, the shift is one-step forward (although you can move X words forward in more advanced scenarios). For example, for the sentence "The cow jumps over the moon". If N=2 (known as bigrams), then the ngrams would be:
* the cow
* cow jumps
* jumps over
* over the
* the moon

We will use NLTK ngrams and word_tokenizer libraries for n-gram feature extraction.

Note: Need to download punkt resource for nltk for work tokenization

In [31]:
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\AMatharaArac/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\AMatharaArac/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

First we define the value for *n*, in n-gram representation.

In [32]:
n = 3

Following n_grams() method will take a sentence and construct a list of n-grams.

In [33]:
def generate_ngrams(text):
  # Convert sentence into list of words
  words = word_tokenize(text)

  # If sentence is too short, return empty list
  if len(words) < 3:
    return []

  n_grams = ngrams(words, n)
  return [' '.join(grams) for grams in n_grams]

In [34]:
txt = "you want to exclude some stop word being getting ignored"
print(generate_ngrams(txt))

['you want to', 'want to exclude', 'to exclude some', 'exclude some stop', 'some stop word', 'stop word being', 'word being getting', 'being getting ignored']


Derive n-grams (n=3) for our dataset.

In [35]:
df['3_grams'] = df['sentence'].apply(generate_ngrams)

Display original sentence and n-grams.

In [36]:
df[['sentence', '3_grams']].head(20)

,sentence,3_grams
0,wow loved place,[wow loved place]
1,crust not good,[crust not good]
2,not tasty texture nasty,"[not tasty texture, tasty texture nasty]"
3,stopped late may bank holiday rick steve recom...,"[stopped late may, late may bank, may bank hol..."
4,selection menu great prices,"[selection menu great, menu great prices]"
5,getting angry want damn pho,"[getting angry want, angry want damn, want dam..."
6,honeslty didnt taste fresh,"[honeslty didnt taste, didnt taste fresh]"
7,potatoes like rubber could tell had made ahead...,"[potatoes like rubber, like rubber could, rubb..."
8,fries great,[]
9,great touch,[]


Based on above results (e.g., record 16) you can see that if there are only 2 words, the 3-grams would result no n-grams.  
Thus, you may try to derive n-grams with *n=2*.

### Bag of words

[Bag of words](https://machinelearningmastery.com/gentle-introduction-bag-words-model/) is a simple text feature extraction mechanism.   
A bag-of-words is a representation of text that describes the occurrence of words within a document. It involves two things:
* A vocabulary of known words.  
* A measure of the presence of known words.  

We will use [CountVectorizer library](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) on sklearn for bag-of-words model creation.

In [37]:
from sklearn.feature_extraction.text import CountVectorizer

You may refer to [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) API for detailed description about the parameters.

In [38]:
# Step 1: Create vectorizer object
bow = CountVectorizer(
    max_features=1000,   # limit number of words
    lowercase=True,      # convert text to lowercase
    ngram_range=(1,1),   # use single words only
    analyzer="word"      # analyze at word level
)

# Step 2: Fit and transform data
# Learn vocabulary + convert text to numeric matrix
X_bow = bow.fit_transform(df['sentence_stemmed'])

# Step 3: Check shape of matrix
print(X_bow.shape)

(2748, 1000)


### Term Frequency - Inverse Document Frequecy (TF-IDF)

[Term frequency–inverse document frequency](https://www.kdnuggets.com/2018/08/wtf-tf-idf.html), is a numerical statistic that is intended to reflect how important a word is to a document in a collection. The tf–idf value increases proportionally to the number of times a word appears in the document and is offset by the number of documents in the corpus that contain the word, which helps to adjust for the fact that some words appear more frequently in general.

We will use [feature extraction module](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) of the sklearn library for this.

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

Construct TF-IDF using the lemmatized senteces.

In [40]:
tf_idf = vectorizer.fit_transform(df['sentence_stemmed'])  # as the text data, we will use lemmatized sentences

Display the list of all the words.

In [41]:
print(vectorizer.get_feature_names_out())

['10' '10oct' '15lb' ... 'zombi' 'zombiestud' 'zombiez']


Here you see there are quite many text that includes a number (digit).  
In one of the pre-processing steps, we removed all the words/text that are only digits, but not combined.  
You might want to remove these as well...  

A comparison of TF-IDF values with respect to lemmatized sentences.

In [42]:
print(df['sentence_lemmatized'].head())

0                                      wow loved place
1                                       crust not good
2                              not tasty texture nasty
3    stopped late may bank holiday rick steve recom...
4                           selection menu great price
Name: sentence_lemmatized, dtype: str


In [43]:
print(tf_idf[:5])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23 stored elements and shape (5, 4132)>
  Coords	Values
  (0, 4088)	0.7549185659179652
  (0, 2134)	0.4820990062440302
  (0, 2675)	0.4446105115838819
  (1, 870)	0.8484433337933565
  (1, 2443)	0.3582623419841641
  (1, 1554)	0.38960493279428215
  (2, 2443)	0.2590688781606039
  (2, 3556)	0.5047727584023991
  (2, 3591)	0.5909620896136903
  (2, 2375)	0.5734558286240188
  (3, 2134)	0.21133869226853622
  (3, 3422)	0.32459788155621533
  (3, 2028)	0.34690375858974987
  (3, 2211)	0.3053549347360731
  (3, 287)	0.39038995129203263
  (3, 1725)	0.39038995129203263
  (3, 2983)	0.39038995129203263
  (3, 3411)	0.357493916424124
  (3, 2893)	0.2266571773812376
  (4, 3132)	0.594161121180882
  (4, 2248)	0.563731151739016
  (4, 1584)	0.34385606899646176
  (4, 2761)	0.45928504705724965


In the feature vector row (e.g., (0, 4843)), the first digit refers to the sentence row (i.e., first datarow).  
The second digit is the index of alphebitically ordered word list.

### Sentiment Analysis

Sentiment analysis is basically the process of determining the attitude or the emotion of the writer, i.e., whether it is positive or negative or neutral.

We will use the Textblob library. The sentiment function of textblob returns the polarity of the sentence, i.e., a float value which lies in the range of [-1,1] where 1 means positive statement and -1 means a negative statement.

In [44]:
from textblob import TextBlob

In [45]:
doc1 = "wow"
TextBlob(doc1).sentiment.polarity

0.1

Derive sentiment of each sentence.

In [46]:
df['sentiment'] = df['sentence_lemmatized'].apply(lambda x: TextBlob(x).sentiment.polarity)

Display original sentece with respect to its sentiment.

In [47]:
print(df[['sentence_lemmatized', 'sentiment']][:40])

                                  sentence_lemmatized  sentiment
0                                     wow loved place   0.400000
1                                      crust not good  -0.350000
2                             not tasty texture nasty  -1.000000
3   stopped late may bank holiday rick steve recom...   0.200000
4                          selection menu great price   0.800000
5                         getting angry want damn pho  -0.500000
6                          honeslty didnt taste fresh   0.300000
7   potato like rubber could tell had made ahead t...   0.000000
8                                           fry great   0.800000
9                                         great touch   0.800000
10                                     service prompt   0.000000
11                                  would not go back   0.000000
12  cashier had care ever had say still ended wayy...   0.000000
13    tried cape cod ravoli chickenwith cranberrymmmm   0.000000
14                   disg

## Text Classification

We will explore few text classification approaches to classify the review data as either positive (1) or negative (0).  
Here we will only use the amazon reviews (1000 reviews) for the workshop. (You may use yelp and imdb review data seperately evaluate the approaches.)

Previously, we conducted all the pre-processing steps to the entire 3 datasets (amazon, yelp and imdb). This for text classification we will filter only the reviews from amazon.

In [48]:
df_amazon = df.loc[df['source'] == 'amazon']

In [49]:
df_amazon.head()

,sentence,label,source,word_count,char_count,avg_word,sentence_stemmed,sentence_lemmatized,3_grams,sentiment
1000,way plug us unless go converter,0,amazon,21,82,2.952381,way plug us unless go convert,way plug u unless go converter,"[way plug us, plug us unless, us unless go, un...",0.00
1001,good case excellent value,1,amazon,4,27,6.000000,good case excel valu,good case excellent value,"[good case excellent, case excellent value]",0.85
1002,great jawbone,1,amazon,4,22,4.750000,great jawbon,great jawbone,[],0.80
1003,tied charger conversations lasting minutesmajo...,0,amazon,11,79,6.272727,tie charger convers last minutesmajor problem,tied charger conversation lasting minutesmajor...,"[tied charger conversations, charger conversat...",0.00
1004,mic great,1,amazon,4,17,3.500000,mic great,mic great,[],0.80


Split train/validation data


In [50]:
from sklearn.model_selection import train_test_split

sentences_train, sentences_test, y_train, y_test = train_test_split(df_amazon['sentence_lemmatized'], df_amazon['label'], test_size=0.3, random_state=2)

### Logistic Regression

We will use the Bag of Words model as text features.

In [51]:
from sklearn.feature_extraction.text import CountVectorizer

Construct the bag of words model.

In [52]:
bow = CountVectorizer(min_df=0.0, lowercase=False)
bow.fit(sentences_train)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


Fit the train and test sentences to transform them to bag-of-word features.

In [53]:
X_train = bow.transform(sentences_train)
X_test  = bow.transform(sentences_test)

Use a logistic regression model for classification.

In [54]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Evaluate the model.

In [55]:
score_train = classifier.score(X_train, y_train)
score_test = classifier.score(X_test, y_test)
print('Train accuracy: {:.2f}%'.format(score_train*100))
print('Test accuracy: {:.2f}%'.format(score_test*100))

Train accuracy: 98.43%
Test accuracy: 81.00%


What can you say about bias and variance of this model?  
- Low bias and high variance

Try few examples

In [56]:
testing = "happy customer"
vector_representation = bow.transform([testing])
prediction = classifier.predict(vector_representation)[0]

if prediction == 1:
   print("Positive Review")
else:
   print("Negative Review")

Positive Review


## Exercise

You may conduct similar classification exercises for yelp and imdb datasets.